In [ ]:
"""
Tampa Zoning Code Scraper v3 — Markdown
===============================================
Extends v2 by reading the pandoc-converted markdown file (Ch26_27.md)
alongside the existing .txt files.

Key improvement over v2:
  Table 4-2 (dimensional standards) is now parsed STRUCTURALLY from the
  md file rather than being hard-coded or read as bare numbers from txt.
  Column headers are preserved as labels, so the TF-IDF vectorizer sees
  tokens like "front_setback: 25" and "max_height: 35" rather than
  standalone numbers that carry no column context.

  This directly addresses the RS-60/RS-50 similarity collapse: both zones
  share identical governing sections (156, 232, 283) in the .txt files,
  but their Table 4-2 rows differ in min_lot_area (6000 vs 5000),
  lot_width (60 vs 50), and front_setback (25 vs 20). With md parsing
  those differences are now expressed as labelled tokens.

What changes vs v2:
  1. MD_PATH config variable — path to Ch26_27.md
  2. parse_table_4_2_from_md() — extracts dimensional standards from md
  3. build_labeled_dim_text() — generates richer labelled strings from
     parsed table rather than hard-coded LABELED_DIM_TEXT dict
  4. SH/NMU table extraction from md — supplements zone-specific
     paragraph extraction with the actual table values
  5. Similarity diagnostics unchanged — same threshold and reporting

Backwards compatible:
  All outputs (zone_text_corpora.csv, zone_features_combined.csv,
  zone_ped_scores.csv) have identical format to v2. Downstream scripts
  (v3–v9, embedding pipeline, DML) require no changes.

Required files:
  Ch26_27.md                — pandoc-converted markdown (NEW)
  Chapter_27_...txt         — existing txt files (unchanged)
  Chapter_26_...txt
  (other chapters as before)
"""

import re, os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# Pandoc-converted markdown preserves table structure that .txt loses.
# Table 4-2 (dimensional standards) parsed structurally from md so TF-IDF
# sees labelled tokens like 'front_setback: 25' instead of bare numbers.
MD_PATH = 'raw_data/Ch26_27.md'      # pandoc md file

# Ch 27 is the core zoning/land development code. Supplementary chapters
# (utilities, streets, affordable housing, fire, business, general) provide
# cross-references and overlay provisions that vary by zone class.
CODE_FILES = [
    'raw_data/all_zoncode/Chapter_27___ZONING_AND_LAND_DEVELOPMENT.txt',
    'raw_data/all_zoncode/Chapter_26___UTILITIES.txt',
    'raw_data/all_zoncode/Chapter_22___STREETS_AND_SIDEWALKS.txt',
    'raw_data/all_zoncode/Chapter_17.5___AFFORDABLE_HOUSING__SUSTAINABILITY__AND_CONCURRENCY_MANAGEMENT_SYSTEM.txt',
    'raw_data/all_zoncode/Chapter_11___FIRE_PREVENTION_AND_PROTECTION.txt',
    'raw_data/all_zoncode/Chapter_6___BUSINESS_REGULATION.txt',
    'raw_data/all_zoncode/Chapter_1___GENERAL_PROVISIONS.txt',
    'raw_data/HILLSBOROUGH_COUNTY___LAND_DEVELOPMENT_CODE.txt',
]

# ─────────────────────────────────────────────────────────────
# 1. PARSE TABLE 4-2 FROM MARKDOWN
# ─────────────────────────────────────────────────────────────

def parse_table_4_2_from_md(md_path):
    """
    Extract Table 4-2 (Schedule of Area, Height, Bulk and Placement
    Regulations) from the pandoc-generated markdown file.

    Pandoc renders Word tables as simple tables with dashed dividers
    and space-padded columns. Table 4-2 rows look like:

      RS-60      6,000     60        6,000       25     7    20/20   7   ---  35
      RS-50      5,000     50        5,000       20     7    20/20   7   ---  35

    Returns dict: {zone_class: {col_name: value, ...}}
    """
    if not os.path.exists(md_path):
        print(f'  WARNING: MD file not found at {md_path}')
        print('  Falling back to hard-coded dimensional text (v2 behaviour)')
        return {}

    with open(md_path, encoding='utf-8') as f:
        md = f.read()

    # Find Table 4-2 block
    # Starts at "TABLE4-2" header, ends at "Table 4-2 Notes"
    t42_start = re.search(r'\*\*TABLE4-2\*\*', md)
    t42_notes = re.search(r'\*\*Table 4-2 Notes:\*\*', md)
    if not t42_start or not t42_notes:
        print('  WARNING: Table 4-2 not found in md file')
        return {}

    table_block = md[t42_start.start():t42_notes.start()]

    # Column names from the header (simplified mapping)
    COLUMNS = [
        'district', 'min_lot_area', 'lot_width', 'dwelling_unit_area',
        'front_setback', 'side_setback', 'rear_setback',
        'corner_setback', 'max_far', 'max_height'
    ]

    # Zone classes to look for in each row
    ALL_BASE_ZONES = [
        'RS-150', 'RS-100', 'RS-75', 'RS-60', 'RS-50',
        'RM-12', 'RM-16', 'RM-18', 'RM-24', 'RM-35', 'RM-50', 'RM-75',
        'RO', 'RO-1', 'OP', 'OP-1', 'CN', 'CG', 'CI', 'IG', 'IH',
    ]

    parsed = {}
    lines = table_block.split('\n')
    for line in lines:
        line = line.strip()
        for zone in ALL_BASE_ZONES:
            if re.match(rf'^{re.escape(zone)}\s+', line):
                rest = line[len(zone):].strip()
                # Strip footnote markers like ^17^, ^12^
                rest = re.sub(r'\^\d+\^', '', rest)
                rest = re.sub(r',', '', rest)
                rest = re.sub(r'\b(n/a|N/A|---)\b', 'NA', rest)
                tokens = re.findall(r'[\d]+(?:/[\d]+)?|NA', rest)
                if len(tokens) < 5:
                    break
                # Detect whether dwelling_unit_area is present at tokens[2]:
                #   - Present (real number >200 or 'NA'): full 8-token row,
                #     front_setback at tokens[3]
                #   - Absent (^17^ stripped to nothing): 7-token row,
                #     front_setback at tokens[2]
                has_dua = (tokens[2] == 'NA' or
                           (tokens[2].isdigit() and int(tokens[2]) > 200))
                front = tokens[3] if (has_dua and len(tokens) > 3) else tokens[2]
                side  = tokens[4] if (has_dua and len(tokens) > 4) else (
                        tokens[3] if len(tokens) > 3 else 'NA')
                # Max height: last purely numeric token
                # But if the original line ends with n/a (IH, RM-75),
                # max_height is unconstrained — mark as NA
                if re.search(r'n/a\s*(?:\^\d+\^)?\s*$', line, re.IGNORECASE):
                    max_height = 'NA'
                else:
                    height_candidates = [t for t in tokens if t.isdigit()]
                    max_height = height_candidates[-1] if height_candidates else 'NA'
                parsed[zone] = {
                    'min_lot_area':  tokens[0],
                    'lot_width':     tokens[1],
                    'front_setback': front,
                    'side_setback':  side,
                    'max_height':    max_height,
                }
                break

    print(f'  Table 4-2: parsed {len(parsed)} zone rows from md')
    return parsed


def build_labeled_dim_text(zone, t42_data):
    """
    Build a labelled dimensional string for a zone using md-parsed Table 4-2
    values where available, falling back to hard-coded values.

    The key difference from v2: values are expressed as labelled phrases
    ("front setback 25 ft") rather than being embedded in generic sentences.
    This gives TF-IDF more discriminatory signal per zone.
    """
    if zone in t42_data:
        d = t42_data[zone]
        parts = [f'{zone} zoning district:']

        def fmt(val, label, unit='ft'):
            if val and val != 'NA':
                # Handle "20/20" style rear/corner combos
                return f'{label} {val} {unit}'
            return None

        field_map = [
            ('min_lot_area',  'minimum lot area',  'sqft'),
            ('lot_width',     'lot width',          'ft'),
            ('front_setback', 'front setback',      'ft'),
            ('side_setback',  'side setback',       'ft'),
            ('max_height',    'maximum height',     'ft'),
        ]
        for col, label, unit in field_map:
            val = d.get(col)
            f = fmt(val, label, unit)
            if f:
                parts.append(f + '.')

        # Add use-type context
        USE_TYPE = {
            'RS-150': 'Large-lot single-family residential.',
            'RS-100': 'Single-family residential.',
            'RS-75':  'Single-family residential.',
            'RS-60':  'Single-family residential, dominant Tampa zone.',
            'RS-50':  'Small-lot single-family residential.',
            'RM-12':  'Low-density multi-family residential, up to 12 units per acre.',
            'RM-16':  'Multi-family residential, up to 16 units per acre.',
            'RM-18':  'Multi-family residential, up to 18 units per acre.',
            'RM-24':  'Medium-density multi-family residential, up to 24 units per acre.',
            'RM-35':  'Medium-high density multi-family residential, up to 35 units per acre.',
            'RM-50':  'High-density multi-family residential, up to 50 units per acre.',
            'RM-75':  'High-density multi-family residential, up to 75 units per acre.',
            'RO':     'Residential-office, low-intensity office compatible with residential.',
            'RO-1':   'Residential-office, higher intensity office.',
            'OP':     'Office park, professional and institutional uses.',
            'OP-1':   'Office park high-intensity, allows limited retail.',
            'CN':     'Commercial neighborhood, limited retail and personal services.',
            'CG':     'Commercial general, variety of retail and commercial service.',
            'CI':     'Commercial intensive, heavy commercial and service uses.',
            'IG':     'Industrial general, light manufacturing and warehousing.',
            'IH':     'Industrial heavy, heavy manufacturing no height limit.',
        }
        if zone in USE_TYPE:
            parts.append(USE_TYPE[zone])

        return ' '.join(parts)

    # Fallback: hard-coded text (same as v2 LABELED_DIM_TEXT)
    FALLBACK = {
        'CBD-1':  'CBD-1 zoning: central business district primary, zero setbacks, no height limit. Downtown Tampa urban core.',
        'CBD-2':  'CBD-2 zoning: central business district secondary, zero setbacks, no height limit. Downtown Tampa fringe.',
        'NMU-16': 'NMU-16 zoning: neighborhood mixed use, build-to line 15-20 ft front, max height 35 ft, transparency 35 percent minimum, no minimum parking.',
        'NMU-24': 'NMU-24 zoning: neighborhood mixed use, build-to line 15-20 ft front, max height 60 ft, transparency 50 percent minimum, no minimum parking.',
        'NMU-35': 'NMU-35 zoning: neighborhood mixed use, build-to line 15-20 ft front, max height 85 ft, transparency 65 percent minimum, no minimum parking.',
        'SH-RS':  'SH-RS zoning: Seminole Heights single-family residential subdistrict, lot width 50 ft, front setback 20 ft, max height 35 ft.',
        'SH-RS-A':'SH-RS-A zoning: Seminole Heights single-family attached, lot width 50 ft, front setback 20 ft, max height 35 ft.',
        'SH-RM':  'SH-RM zoning: Seminole Heights multi-family residential, lot width 50 ft, front setback 15 ft, max height 45 ft.',
        'SH-RO':  'SH-RO zoning: Seminole Heights residential-office, lot width 50 ft, front setback 15 ft, max height 35 ft.',
        'SH-CN':  'SH-CN zoning: Seminole Heights commercial neighborhood, lot width 50 ft, front setback 10 ft, max height 35 ft.',
        'SH-CG':  'SH-CG zoning: Seminole Heights commercial general, lot width 50 ft, front setback 10 ft, max height 45 ft.',
        'SH-CI':  'SH-CI zoning: Seminole Heights commercial intensive, lot width 50 ft, front setback 10 ft, max height 45 ft.',
        'SH-PD':  'SH-PD zoning: Seminole Heights planned development, custom standards.',
        'M-AP-1': 'M-AP-1 zoning: MacDill airport district subzone 1, restricted near active runway.',
        'M-AP-2': 'M-AP-2 zoning: MacDill airport district subzone 2, limited compatible uses.',
        'M-AP-3': 'M-AP-3 zoning: MacDill airport district subzone 3, transitional uses.',
        'M-AP-4': 'M-AP-4 zoning: MacDill airport district subzone 4, outer compatible zone.',
        'CU':     'CU zoning: community use, civic and institutional uses including parks, schools, government buildings.',
        'UC':     'UC zoning: urban community, mixed civic institutional and residential uses.',
        'AS-1':   'AS-1 zoning: airport support, aviation-compatible uses near Tampa International Airport.',
        'PD':     'PD zoning: planned development, custom negotiated standards per project.',
        'PD-A':   'PD-A zoning: planned development alternative, custom standards.',
    }
    return FALLBACK.get(zone, f'{zone} zoning district.')


def extract_sh_nmu_from_md(md_text, zone):
    """
    Extract zone-specific table values from the SH and NMU md sections.
    Returns a structured string with labelled values.
    """
    parts = []

    if zone.startswith('SH-'):
        # Find the specific SH subsection
        sec_map = {
            'SH-RS':   '27-211.2.1',
            'SH-RS-A': '27-211.2.2',
            'SH-RM':   '27-211.2.3',
            'SH-RO':   '27-211.2.4',
            'SH-CN':   '27-211.2.5',
            'SH-CG':   '27-211.6',
            'SH-CI':   '27-211.6',
            'SH-PD':   '27-211.2',
        }
        sec_num = sec_map.get(zone)
        if sec_num:
            pattern = rf'Sec\. {re.escape(sec_num)}\.'
            m = re.search(pattern, md_text)
            if m:
                # Extract up to 3000 chars from this section
                block = md_text[m.start():m.start() + 3000]
                # Extract numeric setback values with context
                setbacks = re.findall(
                    r'(?:BTL|setback|height|width|area)\s*[:\[].*?(\d+\'(?:---\d+\')?)',
                    block, re.IGNORECASE)
                if setbacks:
                    parts.append(f'{zone} development standards: ' +
                                 ' '.join(setbacks[:8]))
                # Extract lot width and area
                lot_w = re.search(r"Lot Width\s+(\d+)'", block)
                lot_a = re.search(r"Lot Area\s+(\d+)\s+SF", block)
                if lot_w:
                    parts.append(f'{zone} lot width {lot_w.group(1)} ft minimum.')
                if lot_a:
                    parts.append(f'{zone} lot area {lot_a.group(1)} sqft minimum.')

    elif zone.startswith('NMU-'):
        # Find NMU-16, NMU-24, or NMU-35 block in Table 212-1
        nmu_sec_m = re.search(r'Sec\. 27-212\.3\.', md_text)
        if nmu_sec_m:
            block = md_text[nmu_sec_m.start():nmu_sec_m.start() + 8000]
            # Find this specific subdistrict's block
            zone_m = re.search(re.escape(f'({zone})'), block)
            if zone_m:
                sub = block[zone_m.start():zone_m.start() + 1500]
                # Extract height, setback, transparency values
                heights = re.findall(r"(\d+)\\?'", sub)
                pcts    = re.findall(r'(\d+)%', sub)
                if heights:
                    parts.append(
                        f'{zone} building placement standards: '
                        f'heights/setbacks {" ".join(heights[:6])} ft.')
                if pcts:
                    parts.append(
                        f'{zone} transparency requirements: '
                        f'{pcts[0]}% minimum facade transparency.')

    return ' '.join(parts)


# ─────────────────────────────────────────────────────────────
# 2. LOAD TXT FILES  (identical to v2)
# ─────────────────────────────────────────────────────────────

ALL_ZONES = [
    'RS-150','RS-100','RS-75','RS-60','RS-50',
    'RM-12','RM-16','RM-18','RM-24','RM-35','RM-50','RM-75',
    'RO-1','RO','OP-1','OP','CN','CG','CI','IG','IH',
    'PD-A','PD',
    'CBD-1','CBD-2',
    'SH-RS-A','SH-RS','SH-RM','SH-RO','SH-CN','SH-CG','SH-CI','SH-PD',
    'NMU-35','NMU-24','NMU-16',
    'M-AP-4','M-AP-3','M-AP-2','M-AP-1',
    'YC-9','YC-8','YC-7','YC-6','YC-5','YC-4','YC-3','YC-2','YC-1',
    'CD-3','CD-2','CD-1',
    'CU','UC','AS-1','RM-24/18',
]

ZONE_SECTIONS = {
    'RS-150':['156','232','283'], 'RS-100':['156','232','283'],
    'RS-75': ['156','232','283'], 'RS-60': ['156','232','283'],
    'RS-50': ['156','232','283'],
    'RM-12': ['156','232','283'], 'RM-16': ['156','232','283'],
    'RM-18': ['156','232','283'], 'RM-24': ['156','232','283'],
    'RM-35': ['156','232','283'], 'RM-50': ['156','232','283'],
    'RM-75': ['156','232','283'],
    'RO':    ['156','164','232','233','283'],
    'RO-1':  ['156','164','232','233','283'],
    'OP':    ['156','232','233','283'], 'OP-1':['156','232','233','283'],
    'CN':    ['156','164','232','233','283'],
    'CG':    ['156','232','233','283'], 'CI':  ['156','232','233','283'],
    'IG':    ['156','283'],             'IH':  ['156','283'],
    'PD':    ['227','283'],             'PD-A':['227','283'],
    'CBD-1': ['181','183','206','283'], 'CBD-2':['181','183','206','283'],
    'SH-RS': ['211.2','211.2.1','211.2.4','211.2.5','211.8','283'],
    'SH-RS-A':['211.2','211.2.1','211.2.2','211.8','283'],
    'SH-RM': ['211.2','211.2.3','211.2.5','211.6','211.8','283'],
    'SH-RO': ['211.2','211.2.4','211.8','283'],
    'SH-CN': ['211.2','211.2.5','211.8','283'],
    'SH-CG': ['211.2','211.6','211.8','283'],
    'SH-CI': ['211.2','211.6','211.8','283'],
    'SH-PD': ['211.2','211.8','283'],
    'NMU-35':['212.2','212.3','212.4','212.6','283'],
    'NMU-24':['212.2','212.3','212.4','212.6','283'],
    'NMU-16':['212.2','212.3','212.4','212.6','283'],
    'M-AP-1':['171','283'], 'M-AP-2':['171','283'],
    'M-AP-3':['171','283'], 'M-AP-4':['171','283'],
    'YC-1':  ['197','198','283'], 'YC-2':['197','198','283'],
    'YC-3':  ['197','198','283'], 'YC-4':['197','198','283'],
    'YC-5':  ['197','198','283'], 'YC-6':['197','198','283'],
    'YC-7':  ['197','198','283'], 'YC-8':['197','198','283'],
    'YC-9':  ['197','198','283'],
    'CD-1':  ['197','199','200','201','206','283'],
    'CD-2':  ['197','199','200','201','206','283'],
    'CD-3':  ['197','199','200','201','206','283'],
    'CU':    ['283'], 'UC':['283'], 'AS-1':['283'],
    'RM-24/18':['156','232','283'],
}

UNIVERSAL_SECTIONS = ['283.16','290.7','155.3.4','283','282','284']
OVERLAY_SECTIONS = {
    'CG':   ['236','238','240','241','243','244'],
    'CI':   ['238','240','241','243'],
    'CN':   ['236','240','241','244'],
    'RM-24':['241','244'], 'RM-35':['241','244'],
    'CBD-1':['236','238','243','244'],
    'CBD-2':['236','238','243','244'],
}
# Shared sections are deduplicated by extracting only zone-specific paragraphs
# (via extract_zone_specific_paragraphs) rather than appending the full section.
# Without this, SH- and NMU- zones would have near-identical corpora.
SHARED_SECTIONS = {
    '211.2','211.2.1','211.2.2','211.2.3','211.2.4','211.2.5',
    '211.6','211.7','211.8','212.2','212.4','212.6','197','198','171',
}
NMU_TABLE_SECTION = '212.3'

RESIDENTIAL    = ['RS-150','RS-100','RS-75','RS-60','RS-50',
                  'RM-12','RM-16','RM-18','RM-24','RM-35','RM-50','RM-75',
                  'RO','RO-1','SH-RS','SH-RS-A','SH-RM','SH-RO','SH-PD',
                  'YC-2','YC-4','YC-8','YC-9']
NON_RESIDENTIAL = ['CG','CI','CN','OP','OP-1','IG','IH','CBD-1','CBD-2',
                   'CD-1','CD-2','CD-3','NMU-35','NMU-24','NMU-16',
                   'SH-CG','SH-CI','SH-CN','YC-1','YC-3','YC-5','YC-6','YC-7']
ZONE_CONTEXT   = {z: ('Residential'    if z in RESIDENTIAL else
                      'Non-Residential' if z in NON_RESIDENTIAL else 'Other/PD')
                  for z in ALL_ZONES}


# Extract 600-char window around each zone mention: captures the full regulation
# paragraph without pulling in adjacent zones' provisions from shared sections.
def extract_zone_specific_paragraphs(section_body, zone_code, window=600):
    parts = []
    for m in re.finditer(re.escape(zone_code), section_body, re.IGNORECASE):
        start = max(0, m.start() - 80)
        end   = min(len(section_body), m.end() + window)
        parts.append(section_body[start:end])
    return ' '.join(parts)


def extract_nmu_block(section_body, zone_code):
    nmu_order = ['NMU-16','NMU-24','NMU-35']
    if zone_code not in nmu_order:
        return ''
    idx   = nmu_order.index(zone_code)
    start = section_body.find(zone_code)
    if start < 0:
        return ''
    end = (section_body.find(nmu_order[idx + 1])
           if idx + 1 < len(nmu_order) else len(section_body))
    if end < 0:
        end = len(section_body)
    return section_body[start:end]


def clean_text(t):
    t = re.sub(r'\t', ' ', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


# ─────────────────────────────────────────────────────────────
# 3. LOAD AND PARSE TXT SECTIONS
# ─────────────────────────────────────────────────────────────
section_pattern = re.compile(r'Sec\.\s+27-([\d\w\.\-]+)\.', re.MULTILINE)
sections  = {}
all_lines = []

print('Loading code files...')
for fname in CODE_FILES:
    if not os.path.exists(fname):
        print(f'  SKIPPED: {fname}')
        continue
    with open(fname, encoding='utf-8-sig') as f:
        raw = f.read()
    text = raw.replace('\r\n', '\n').replace('\r', '\n')
    all_lines.extend(text.split('\n'))
    splits = section_pattern.split(text)
    n = 0
    for i in range(1, len(splits), 2):
        sec_num = splits[i]
        body    = splits[i + 1] if i + 1 < len(splits) else ''
        if sec_num not in sections:
            sections[sec_num] = body
        else:
            sections[sec_num] += '\n' + body
        n += 1
    print(f'  {fname.split("/")[-1][:55]}: {n} sections')

print(f'Total sections: {len(sections)}  |  Lines: {len(all_lines):,}')

# Zone-name line index
zone_line_index = {z: [] for z in ALL_ZONES}
for line in all_lines:
    stripped = line.strip()
    if len(stripped) < 25:
        continue
    for zone in ALL_ZONES:
        if zone in line and re.search(
                rf'\b{re.escape(zone)}\b', line, re.IGNORECASE):
            zone_line_index[zone].append(stripped)

# ─────────────────────────────────────────────────────────────
# 4. LOAD MD FILE AND PARSE TABLE 4-2
# ─────────────────────────────────────────────────────────────
print('\nLoading markdown file...')
md_text = ''
if os.path.exists(MD_PATH):
    with open(MD_PATH, encoding='utf-8') as f:
        md_text = f.read()
    print(f'  Loaded: {MD_PATH}  ({len(md_text):,} chars)')
else:
    print(f'  NOT FOUND: {MD_PATH} — falling back to v2 labelled text')

t42_data = parse_table_4_2_from_md(MD_PATH)

# ─────────────────────────────────────────────────────────────
# 5. BUILD ZONE CORPORA  (v2 logic + md table injection)
# ─────────────────────────────────────────────────────────────
print('\nBuilding zone corpora (v3 — md-aware)...')

corpora = {}
for zone in ALL_ZONES:
    parts = []

    # 0. Labelled dimensional text from md Table 4-2 (richer than v2)
    dim_text = build_labeled_dim_text(zone, t42_data)
    parts.append(dim_text)

    # 0b. SH/NMU md table extraction (structural values from md)
    if zone.startswith('SH-') or zone.startswith('NMU-'):
        md_vals = extract_sh_nmu_from_md(md_text, zone)
        if md_vals:
            parts.append(md_vals)

    # 1. Primary governing sections from txt
    for s in ZONE_SECTIONS.get(zone, []):
        body = sections.get(s, '')
        if not body:
            continue
        if s == NMU_TABLE_SECTION:
            block = extract_nmu_block(body, zone)
            if block:
                parts.append(block)
        elif s in SHARED_SECTIONS:
            passage = extract_zone_specific_paragraphs(body, zone)
            parts.append(passage if passage else body[:2000])
        else:
            parts.append(body[:5000])

    # 2. Overlay sections
    for s in OVERLAY_SECTIONS.get(zone, []):
        body = sections.get(s, '')
        if body:
            parts.append(body[:2000])

    # 3. Universal supplemental sections
    for s in UNIVERSAL_SECTIONS:
        body = sections.get(s, '')
        if body:
            parts.append(body[:1000])

    # 4. Zone-name lines
    parts.extend(zone_line_index[zone][:80])

    corpora[zone] = clean_text(' '.join(parts))

# Print corpus richness
print(f'\n{"Zone":<12} {"Chars":>7} {"Zone lines":>10}  Context')
print('─' * 45)
for zone in ALL_ZONES:
    print(f'{zone:<12} {len(corpora[zone]):>7,} '
          f'{len(zone_line_index[zone]):>10}  {ZONE_CONTEXT[zone]}')

# Save corpora
corpus_df = pd.DataFrame([
    {'zone_class': z, 'context': ZONE_CONTEXT[z],
     'corpus_chars': len(corpora[z]), 'corpus': corpora[z]}
    for z in ALL_ZONES
])
corpus_df.to_csv('zone_text_corpora.csv', index=False)
print(f'\nSaved: zone_text_corpora.csv')

# ─────────────────────────────────────────────────────────────
# 6. SIMILARITY DIAGNOSTICS
# ─────────────────────────────────────────────────────────────
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    texts = [corpora[z] for z in ALL_ZONES]
    tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                            min_df=1, sublinear_tf=True)
    mat   = tfidf.fit_transform(texts)
    sim   = cosine_similarity(mat)

    print('\n── Corpus Cosine Similarity Diagnostics (v3 — md-aware) ──')
    high = [(ALL_ZONES[i], ALL_ZONES[j], sim[i,j])
            for i in range(len(ALL_ZONES))
            for j in range(i+1, len(ALL_ZONES))
            if sim[i,j] > 0.90]
    high.sort(key=lambda x: -x[2])
    print(f'  Pairs with similarity > 0.90: {len(high)}')
    print(f'  (v2 had 36; target is <20)')
    for a, b, s in high[:15]:
        print(f'    {a:<10} ↔ {b:<10}  {s:.4f}')
    if len(high) > 15:
        print(f'    ... and {len(high)-15} more')
except ImportError:
    print('  sklearn not available — skipping diagnostics')

# ─────────────────────────────────────────────────────────────
# 7. PEDESTRIAN ORIENTATION SCORING  (identical to v2)
# ─────────────────────────────────────────────────────────────
CRITERIA = {
    'sidewalk_req':      [r'sidewalk', r'pedestrian\s*path', r'walking\s*path'],
    'frontage_std':      [r'build.to\s*zone', r'build.to.line', r'frontage\s*require',
                          r'building\s*streetwall', r'zero\s*setback', r'front\s*yard.*waiv'],
    'active_ground_flr': [r'ground.floor\s*(?:retail|commercial|active|use)',
                          r'active\s*(?:ground|street).floor',
                          r'ground.floor\s*transparen'],
    'mixed_use_perm':    [r'mixed.use', r'mix(?:ed)?\s*of\s*uses',
                          r'residential.*commercial', r'commercial.*residential'],
    'ped_scale_lang':    [r'pedestrian.(?:friendly|scale|oriented|priority|environment)',
                          r'walkab', r'human.scale'],
    'bicycle_prov':      [r'bicycle', r'bike\s*lane', r'cycling', r'bike\s*rack'],
    'transit_orient':    [r'transit.oriented', r'transit\s*(?:stop|station|access)',
                          r'public\s*transportation', r'bus\s*stop'],
    'reduced_parking':   [r'parking\s*(?:reduction|waiver|exempt)',
                          r'reduce.*parking', r'no\s*(?:minimum\s*)?parking',
                          r'shared\s*parking'],
    'landscaping_req':   [r'landscap', r'street\s*tree', r'tree\s*canopy',
                          r'green\s*space', r'vegetation'],
    'human_scale_design':[r'fa[cç]ade', r'fenestration', r'transparency',
                          r'architectural\s*(?:detail|feature|element)',
                          r'street.level\s*design'],
}

print('\nScoring pedestrian orientation...')
ped_records = []
for zone in ALL_ZONES:
    corpus = corpora[zone]
    scores = {}
    for criterion, patterns in CRITERIA.items():
        scores[criterion] = int(any(
            re.search(p, corpus, re.IGNORECASE) for p in patterns
        ))
    scores['ped_score_total'] = sum(scores[c] for c in CRITERIA)
    scores['ped_score_norm']  = scores['ped_score_total'] / 10.0
    row = {'zone_class': zone}
    row.update(scores)
    ped_records.append(row)
    bar = ''.join('█' if scores[c] else '░' for c in CRITERIA)
    print(f'  {zone:<12} {scores["ped_score_total"]}/10  {bar}  ({len(corpus):,} chars)')

ped_df = pd.DataFrame(ped_records)

# ─────────────────────────────────────────────────────────────
# 8. DIMENSIONAL STANDARDS TABLE  (identical to v2)
# ─────────────────────────────────────────────────────────────
TABLE_4_2 = {
    'RS-150': dict(min_lot_area=15000, lot_width=100, front_setback=30, side_setback=15, rear_setback=15, max_height=35),
    'RS-100': dict(min_lot_area=10000, lot_width=100, front_setback=25, side_setback=7,  rear_setback=15, max_height=35),
    'RS-75':  dict(min_lot_area=7500,  lot_width=75,  front_setback=25, side_setback=7,  rear_setback=15, max_height=35),
    'RS-60':  dict(min_lot_area=6000,  lot_width=60,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=35),
    'RS-50':  dict(min_lot_area=5000,  lot_width=50,  front_setback=20, side_setback=7,  rear_setback=7,  max_height=35),
    'RM-12':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=35),
    'RM-16':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=35),
    'RM-18':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=35),
    'RM-24':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=60),
    'RM-35':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=120),
    'RM-50':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=200),
    'RM-75':  dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=7,  max_height=None),
    'RO':     dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=15, max_height=35),
    'RO-1':   dict(min_lot_area=5000,  lot_width=50,  front_setback=25, side_setback=7,  rear_setback=15, max_height=35),
    'OP':     dict(min_lot_area=10000, lot_width=60,  front_setback=25, side_setback=10, rear_setback=25, max_height=60),
    'OP-1':   dict(min_lot_area=10000, lot_width=60,  front_setback=20, side_setback=10, rear_setback=20, max_height=200),
    'CN':     dict(min_lot_area=5000,  lot_width=60,  front_setback=20, side_setback=10, rear_setback=20, max_height=35),
    'CG':     dict(min_lot_area=10000, lot_width=75,  front_setback=10, side_setback=10, rear_setback=10, max_height=45),
    'CI':     dict(min_lot_area=10000, lot_width=100, front_setback=10, side_setback=0,  rear_setback=10, max_height=45),
    'IG':     dict(min_lot_area=5000,  lot_width=50,  front_setback=10, side_setback=0,  rear_setback=10, max_height=60),
    'IH':     dict(min_lot_area=5000,  lot_width=50,  front_setback=10, side_setback=0,  rear_setback=10, max_height=None),
    'CBD-1':  dict(min_lot_area=None,  lot_width=None, front_setback=0, side_setback=0,  rear_setback=0,  max_height=None),
    'CBD-2':  dict(min_lot_area=None,  lot_width=None, front_setback=0, side_setback=0,  rear_setback=0,  max_height=None),
    'SH-RS':  dict(min_lot_area=5000,  lot_width=50,  front_setback=20, side_setback=5,  rear_setback=5,  max_height=35),
    'SH-RS-A':dict(min_lot_area=5000,  lot_width=50,  front_setback=20, side_setback=5,  rear_setback=5,  max_height=35),
    'SH-RM':  dict(min_lot_area=5000,  lot_width=50,  front_setback=15, side_setback=5,  rear_setback=5,  max_height=45),
    'SH-RO':  dict(min_lot_area=5000,  lot_width=50,  front_setback=15, side_setback=5,  rear_setback=5,  max_height=35),
    'SH-CN':  dict(min_lot_area=5000,  lot_width=50,  front_setback=10, side_setback=0,  rear_setback=5,  max_height=35),
    'SH-CG':  dict(min_lot_area=5000,  lot_width=50,  front_setback=10, side_setback=0,  rear_setback=5,  max_height=45),
    'SH-CI':  dict(min_lot_area=5000,  lot_width=50,  front_setback=10, side_setback=0,  rear_setback=5,  max_height=45),
    'SH-PD':  dict(min_lot_area=None,  lot_width=None, front_setback=None, side_setback=None, rear_setback=None, max_height=None),
    'NMU-35': dict(min_lot_area=None,  lot_width=None, front_setback=0, side_setback=0,  rear_setback=0,  max_height=None),
    'NMU-24': dict(min_lot_area=None,  lot_width=None, front_setback=0, side_setback=0,  rear_setback=0,  max_height=None),
    'NMU-16': dict(min_lot_area=None,  lot_width=None, front_setback=0, side_setback=0,  rear_setback=0,  max_height=None),
}
for zone in ALL_ZONES:
    if zone not in TABLE_4_2:
        TABLE_4_2[zone] = dict(min_lot_area=None, lot_width=None,
                               front_setback=None, side_setback=None,
                               rear_setback=None, max_height=None)

dim_df = pd.DataFrame.from_dict(TABLE_4_2, orient='index').reset_index()
dim_df.columns = ['zone_class'] + list(dim_df.columns[1:])

# ─────────────────────────────────────────────────────────────
# 9. MERGE AND SAVE
# ─────────────────────────────────────────────────────────────
combined = ped_df.merge(dim_df, on='zone_class', how='left')
combined['ped_score_norm'] = combined['ped_score_total'] / 10.0

combined.to_csv('zone_features_combined.csv', index=False)
ped_df.to_csv('zone_ped_scores_v2.csv', index=False)
dim_df.to_csv('zone_dimensional_standards.csv', index=False)

print('\nFiles saved:')
print('  zone_text_corpora.csv          — v3 md-aware corpora')
print('  zone_features_combined.csv     — primary classifier input')
print('  zone_ped_scores_v2.csv')
print('  zone_dimensional_standards.csv')
print('\n✓ Scraper v3 complete.')

In [ ]:
#the final file names are df (origin and destination in the county area),destination county, and origin county. df is the only one with geometry